In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from torch.nn import MultiheadAttention
import numpy as np

# Sanity constants
name = "attn_cmp"
batch = 1
seq_len = 4
n_dims = 16
n_heads = 4
head_dim = n_dims // n_heads
dtype = torch.float16
device = "cuda"
seed = 42

torch.manual_seed(seed)

In [3]:
# Create dummy input
x = torch.randn(seq_len, batch, n_dims, dtype=dtype, device=device)  # shape: [T, B, D]

# Extract components
head_dim = n_dims // n_heads

# 1. Setup PyTorch Attention
attn = torch.nn.MultiheadAttention(embed_dim=n_dims, num_heads=n_heads, bias=True, dropout=0.0, batch_first=False, dtype=dtype, device=device)

In [4]:
from lkeravnos import Transformer

# Construct CUDA transformer
Transformer.construct(name, batch_size=batch, sequence_length=seq_len, num_dims=n_dims, num_heads=n_heads, verbose=True)

> keravnos (main.cu:24) transformer 'attn_cmp' constructed.
> keravnos (transformer.cu:89) allocating 1541120 bytes...
> keravnos [cuda] (memory.cu:9) allocated 1541120 bytes to memory address 0x701600000
> keravnos [cuda] (memory.cu:64) copied 176 bytes from host address 0xbff6fec500 to device address 0x701600000
> keravnos (transformer.cu:94) transformer allocated device memory.
------------------------
Total Bytes            : 1541120 bytes
Batch Size             : 1
Sequence Length        : 4
Vocab Size             : 48000
Embedding Dim          : 16
Number of Heads        : 4
Header Size            : 176 bytes
Token Embedding        : 1536000 bytes
Positional Embedding   : 128 bytes
Token IDs              : 16 bytes
Input Embedding        : 128 bytes
Dropout Mask           : 128 bytes
QKV Projection         : 1536 bytes
QKV Matrix             : 384 bytes
QKV Bias               : 96 bytes
Attention Scores       : 128 bytes
Context Layer          : 128 bytes
Output Projection      :

In [5]:
def half_tensor_to_uint16_numpy(tensor: torch.Tensor) -> np.ndarray:
    """Convert torch.float16 Tensor to np.uint16 using raw memory."""
    return tensor.cpu().numpy().view(np.uint16)

with torch.no_grad():
  qkv_weight = attn.in_proj_weight  # shape: [3D, D]
  qkv_bias = attn.in_proj_bias      # shape: [3D]
  out_weight = attn.out_proj.weight  # [D, D]
  out_bias = attn.out_proj.bias      # [D]

  # .reshape(...).contiguous() just to be safe
  qkv_weight_np = half_tensor_to_uint16_numpy(qkv_weight.reshape(3, n_dims, n_dims).contiguous())
  qkv_bias_np = half_tensor_to_uint16_numpy(qkv_bias.reshape(3, n_dims).contiguous())
  out_weight_np = half_tensor_to_uint16_numpy(out_weight.contiguous())
  out_bias_np = half_tensor_to_uint16_numpy(out_bias.contiguous())

  Transformer.edit_tensor(name, "qkv_proj", qkv_weight_np)
  Transformer.edit_tensor(name, "qkv_proj_bias", qkv_bias_np)
  Transformer.edit_tensor(name, "out_proj", out_weight_np)
  Transformer.edit_tensor(name, "out_proj_bias", out_bias_np)

In [6]:
token_embed = x.transpose(0, 1).contiguous().view(batch * seq_len, n_dims).cpu().numpy().view(np.uint16)
Transformer.edit_tensor(name, "input_embed", token_embed, verbose=True)

> keravnos [cuda] (memory.cu:9) allocated 128 bytes to memory address 0x701778400
> keravnos [cuda] (memory.cu:64) copied 128 bytes from host address 0x4f3b0050080 to device address 0x701778400
> keravnos [cuda] (memory.cu:80) copied 176 bytes from device address 0x701600000 to host address 0xbff6fecb50
> keravnos [cuda] (memory.cu:36) deallocated memory address 0x701778400
> keravnos (main.cu:126) edit completed for tensor id 'input_embed' of transformer 'attn_cmp'.


In [7]:
Transformer.causal_self_attention(name, use_bias=True, dropout=0.0, seed=42, verbose=True)

> keravnos [cuda] (memory.cu:80) copied 176 bytes from device address 0x701600000 to host address 0xbff6fece50


In [8]:
qkv_cuda = Transformer.get_tensor(name, "qkv_matrix", True)  # shape: [B, T, 3, D]
qkv_cuda = torch.from_numpy(qkv_cuda.view(np.float16)).to(dtype).to(device)
print(qkv_cuda)

> keravnos [cuda] (memory.cu:80) copied 176 bytes from device address 0x701600000 to host address 0xbff6fecb40
> keravnos [cuda] (memory.cu:9) allocated 384 bytes to memory address 0x701778400
> keravnos [cuda] (memory.cu:80) copied 384 bytes from device address 0x701778400 to host address 0x2a9b88afef0
> keravnos [cuda] (memory.cu:36) deallocated memory address 0x701778400
tensor([[[[ 6.9214e-02,  2.0239e-01, -3.2562e-02, -4.9225e-02,  1.2646e-01,
            8.7646e-01,  9.2969e-01,  4.1626e-01,  1.4678e+00, -9.2236e-01,
           -6.8701e-01,  5.8936e-01,  1.9641e-01,  1.0986e+00,  2.3389e-01,
           -9.8877e-01],
          [-3.7524e-01,  7.7295e-01, -7.8979e-02, -2.2656e-01, -5.3589e-02,
           -4.3262e-01, -4.5459e-01,  4.9927e-01, -1.1045e+00,  1.1859e-01,
           -4.0100e-02, -9.2773e-01, -7.4658e-01,  4.8755e-01, -1.0176e+00,
            1.7883e-01],
          [-8.9941e-01, -1.2720e-01, -5.5615e-01, -1.6953e+00,  7.2266e-01,
           -1.7029e-01, -1.3066e+00,  1.6

In [9]:
qkv_ref = torch.nn.functional.linear(x, qkv_weight, qkv_bias)
qkv_ref = qkv_ref.view(batch, seq_len, 3, n_dims).contiguous()  # [B, T, 3, D]
print(qkv_ref)

tensor([[[[ 6.9214e-02,  2.0239e-01, -3.2562e-02, -4.9225e-02,  1.2646e-01,
            8.7646e-01,  9.2969e-01,  4.1626e-01,  1.4678e+00, -9.2236e-01,
           -6.8701e-01,  5.8936e-01,  1.9641e-01,  1.0986e+00,  2.3389e-01,
           -9.8877e-01],
          [-3.7524e-01,  7.7295e-01, -7.8979e-02, -2.2656e-01, -5.3589e-02,
           -4.3262e-01, -4.5459e-01,  4.9927e-01, -1.1045e+00,  1.1859e-01,
           -4.0100e-02, -9.2773e-01, -7.4658e-01,  4.8755e-01, -1.0176e+00,
            1.7883e-01],
          [-8.9941e-01, -1.2720e-01, -5.5615e-01, -1.6953e+00,  7.2266e-01,
           -1.7029e-01, -1.3066e+00,  1.6687e-01, -1.8176e-01,  2.0361e-01,
           -1.8970e-01, -6.8408e-01, -8.2336e-02, -1.1543e+00, -9.6558e-02,
           -1.8398e+00]],

         [[ 1.1157e-01,  1.6084e+00, -1.6572e+00,  9.2041e-01, -1.6125e-01,
           -4.0967e-01, -4.5557e-01,  1.6191e+00, -4.6680e-01,  2.1118e-01,
            1.2952e-01, -1.1016e+00,  7.8369e-01, -1.7737e-01, -5.7861e-01,
           

In [10]:
diff = torch.abs(qkv_ref - qkv_cuda)
max_diff = diff.max()
print("Max diff:", max_diff.item())

if max_diff < 1e-2:
    print("✅ QKV projection matches!")
else:
    print("❌ QKV projection mismatch!")


Max diff: 0.0
✅ QKV projection matches!


In [11]:
attn.eval()
with torch.no_grad():
  qkv_ref_pytorch, _ = attn(x, x, x, need_weights=True, average_attn_weights=False)

In [12]:
# Grab the raw Q, K for inspection
qkv = torch.nn.functional.linear(x, qkv_weight, qkv_bias)  # [T, B, 3D]
qkv = qkv.view(seq_len, batch, 3, n_heads, head_dim).permute(2, 1, 3, 0, 4)  # [3, B, H, T, D]
q, k, v = qkv[0], qkv[1], qkv[2]  # each: [B, H, T, D]

# Compute attention scores manually (before softmax)
q_scaled = q / head_dim**0.5
attn_scores_ref = torch.einsum("bhid,bhjd->bhij", q_scaled, k)  # [B, H, T, T]

# Apply causal mask if needed
mask = torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool), diagonal=1)
attn_scores_ref = attn_scores_ref.masked_fill(mask, float("-inf"))

# Apply softmax
attn_probs_ref = torch.softmax(attn_scores_ref, dim=-1)  # [B, H, T, T]

# Apply dropout
dropout_p = 0.0  # or 0.1 if you want to test that too
if dropout_p > 0.0:
    attn_probs_ref = torch.nn.functional.dropout(attn_probs_ref, p=dropout_p, training=True)

In [13]:
attn_probs_ref

tensor([[[[1.0000, 0.0000, 0.0000, 0.0000],
          [0.9258, 0.0740, 0.0000, 0.0000],
          [0.3928, 0.1610, 0.4463, 0.0000],
          [0.3010, 0.1115, 0.4111, 0.1764]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5156, 0.4841, 0.0000, 0.0000],
          [0.3638, 0.2064, 0.4297, 0.0000],
          [0.2556, 0.3545, 0.2100, 0.1798]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.7007, 0.2991, 0.0000, 0.0000],
          [0.2114, 0.2404, 0.5483, 0.0000],
          [0.2274, 0.1766, 0.2998, 0.2961]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4875, 0.5127, 0.0000, 0.0000],
          [0.2617, 0.3845, 0.3538, 0.0000],
          [0.3350, 0.2705, 0.1940, 0.2006]]]], device='cuda:0',
       dtype=torch.float16, grad_fn=<SoftmaxBackward0>)

In [14]:
# Fetch CUDA side output
attn_probs_cuda = Transformer.get_tensor(name, "attention_scores")
attn_probs_cuda = torch.from_numpy(attn_probs_cuda.view(np.float16)).to(dtype).to(device)
attn_probs_cuda

tensor([[[[1.0000, 0.0000, 0.0000, 0.0000],
          [0.8716, 0.1283, 0.0000, 0.0000],
          [0.1941, 0.4622, 0.3438, 0.0000],
          [0.3250, 0.1442, 0.2178, 0.3130]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4863, 0.5137, 0.0000, 0.0000],
          [0.2683, 0.5991, 0.1327, 0.0000],
          [0.1554, 0.5273, 0.1672, 0.1499]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.6675, 0.3323, 0.0000, 0.0000],
          [0.3687, 0.3564, 0.2751, 0.0000],
          [0.3262, 0.1948, 0.1060, 0.3730]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4163, 0.5835, 0.0000, 0.0000],
          [0.4019, 0.3394, 0.2585, 0.0000],
          [0.2351, 0.3191, 0.2128, 0.2329]]]], device='cuda:0',
       dtype=torch.float16)

In [15]:
# Sum over keys for each query: should be ≈1
for t in range(head_dim): print(attn_probs_cuda[0, 3, t, :].sum())

tensor(1., device='cuda:0', dtype=torch.float16)
tensor(1., device='cuda:0', dtype=torch.float16)
tensor(1., device='cuda:0', dtype=torch.float16)
tensor(1., device='cuda:0', dtype=torch.float16)


In [16]:
torch.allclose(attn_probs_ref, attn_probs_cuda, rtol=1e-2, atol=1e-3)

False

In [17]:
(torch.abs(attn_probs_cuda - attn_probs_ref) > 1e-2).sum()

tensor(36, device='cuda:0')